# V6 Feature Prep — Q1 Features + Q2 Target

**Tarih:** 2026-05-12 (güncellendi: target ≥3 eşiği)
**Amaç:** ML V6 için Q1 verisinden 12 feature + Q2 verisinden target hesaplayıp CSV export.

## Pipeline
- **Q1 (Oca-Mar 2025):** Feature hesabı (12 feature)
- **Q2 (Nis-Haz 2025):** Target hesabı (binary, **≥3 ciddi arıza eşiği**)
- **Çıktı:** `features_final_v6_q1.csv` (3,508 araç × 27 kolon)

## 12 Feature Listesi
| # | Feature | Tip | Kaynak |
|---|---|---|---|
| 1 | yas | Statik | MODELYILI |
| 2 | egim_maruziyet | Q1 | arac_gunluk_hatlar + hat_elevation |
| 3 | garaj_sistem_lift | Q1 | ariza_model GARAJ × ARIZAUSTKODTANIM |
| 4 | garaj_marka_lift | Q1 | ariza_model GARAJ × MARKA |
| 5 | yakit_turu_cng | Statik | YAKITTURU |
| 6 | verimsizlik_skoru | Statik | yas × tuketim |
| 7 | hat_zorluk | Q1 | egim + uzunluk + ariza + trafik bilesik |
| 8 | cascade_risk_skor | Q1 | 24h tekrar ariza orani |
| 9 | farkli_sofor_sayisi | Q1 | nunique SOFOR_SICILNO |
| 10 | dur_kalk_index | Q1 | KAPASITE × sefer / ort_sefer_km |
| 11 | ariza_q1 | Q1 | count ariza |
| 12 | gecmis_ciddi_oran | Q1 | mean ciddi_ariza |

## Target (V6 Final)
- `target_q2 = (q2_ciddi_n >= 3).astype(int)` — Q2'de **3+ ciddi arıza** = 1, aksi 0
- Pozitif oran: %54 (dengeli) — TARGET_KARSILASTIRMA.ipynb seçimi (C tanımı)
- Eski "en az 1 ciddi" tanımı (%86 pozitif) değiştirildi → sınıf dengesizliği problemi çözüldü

## Notlar
- Bilinmiyor YAKITTURU düzeltmesi (CNG modeli ise CNG, değilse MOTORIN)
- ELEKTRIK, Yedek, Kayıtsız araç filtre dışında
- Q1'de boş araç: NaN → 0 veya median (feature'a göre)
- ÖHO/KOOP zaten ariza_model.csv'de yok

---
## 1. Veri Yukleme + Q1/Q2 Split + Filtre


In [1]:
# BOLUM 1: Veri yukleme
import pandas as pd
import numpy as np
import json as _json
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Ana veri
df = pd.read_csv("../panel_data/temiz_veri/ariza_model.csv", low_memory=False)
df["OLAYTARIHI"] = pd.to_datetime(df["OLAYTARIHI"], format="mixed")
print(f"Ham veri: {len(df):,} ariza, {df['KAPINO'].nunique():,} arac")

# Bilinmiyor YAKITTURU duzeltmesi
mask_b = df["YAKITTURU"]=="Bilinmiyor"
df.loc[mask_b & df["MODEL"].str.contains("CNG", na=False), "YAKITTURU"] = "CNG"
df.loc[mask_b & ~df["MODEL"].str.contains("CNG", na=False), "YAKITTURU"] = "MOTORIN"
df = df[df["YAKITTURU"].isin(["MOTORIN","CNG"])].copy()
print(f"Filtrelemis: {len(df):,} ariza, {df['KAPINO'].nunique():,} arac")

# Q1 / Q2 split
SPLIT_DATE = pd.Timestamp("2025-04-01")
df_q1 = df[df["OLAYTARIHI"] < SPLIT_DATE].copy()
df_q2 = df[df["OLAYTARIHI"] >= SPLIT_DATE].copy()
print(f"\nQ1 (Oca-Mar): {len(df_q1):,} ariza, {df_q1['KAPINO'].nunique():,} arac")
print(f"Q2 (Nis-Haz): {len(df_q2):,} ariza, {df_q2['KAPINO'].nunique():,} arac")

# Arac master listesi (Q1 + Q2'de gorunmus tum araclar)
TUM_ARAC = df["KAPINO"].unique()
print(f"\nTum unique arac: {len(TUM_ARAC):,}")


Ham veri: 58,559 ariza, 3,509 arac
Filtrelemis: 58,557 ariza, 3,508 arac

Q1 (Oca-Mar): 27,581 ariza, 3,445 arac
Q2 (Nis-Haz): 30,976 ariza, 3,452 arac

Tum unique arac: 3,508


---
## 2. Statik Feature'lar: yas, yakit_turu_cng, verimsizlik_skoru

Arac kimligine bagli statik bilgiler (Q1/Q2 fark etmez).


In [2]:
# BOLUM 2: Statik feature'lar
# Arac meta (KAPINO bazinda, ilk degeri al)
arac = df.groupby("KAPINO").agg(
    MARKA=("MARKA","first"),
    MODEL=("MODEL","first"),
    MODELYILI=("MODELYILI","first"),
    ARACCINSI=("ARACCINSI","first"),
    KAPASITE=("KAPASITE","first"),
    YAKITTURU=("YAKITTURU","first"),
    GARAJ=("GARAJ","first"),
).reset_index()
print(f"Arac master: {len(arac):,}")

# 1. yas
arac["yas"] = 2025 - arac["MODELYILI"]
print(f"\nyas: min={arac['yas'].min()}, max={arac['yas'].max()}, mean={arac['yas'].mean():.1f}")

# 2. yakit_turu_cng
arac["yakit_turu_cng"] = (arac["YAKITTURU"]=="CNG").astype(int)
print(f"yakit_turu_cng: CNG={arac['yakit_turu_cng'].sum()} ({arac['yakit_turu_cng'].mean()*100:.1f}%)")

# 3. verimsizlik_skoru = yas_norm × tuketim_norm / 100
tuketim = pd.DataFrame([
    ("OTOKAR","KENT 290LF",40),("OTOKAR","KENT XL",60),
    ("MERCEDES","CITARO 0530",39),("MERCEDES","CITARO 0530 G",58),
    ("MERCEDES","CONECTO G",62),("MERCEDES","CONECTO",42),
    ("MERCEDES","CAPACITY",65),
    ("BMC","PROCITY TR",41),("BMC","PROCITY",41),
    ("KARSAN","AVANCITY S PLUS",58),("KARSAN","AVANCITY CNG",52),
    ("TEMSA","AVENUE LF CNG",50),
    ("AKIA","ULTRA LF12",40),("AKIA","LF25",60),
], columns=["MARKA","MODEL","tuketim_100km"])
arac = arac.merge(tuketim, on=["MARKA","MODEL"], how="left")
arac["tuketim_100km"] = arac["tuketim_100km"].fillna(arac["tuketim_100km"].median())

def norm_0_100(s):
    if s.max() > s.min():
        return ((s - s.min())/(s.max() - s.min()) * 100).round(1)
    return pd.Series(0.0, index=s.index)
yas_norm = norm_0_100(arac["yas"])
tuk_norm = norm_0_100(arac["tuketim_100km"])
arac["verimsizlik_skoru"] = (yas_norm * tuk_norm / 100).round(2)
print(f"verimsizlik_skoru: mean={arac['verimsizlik_skoru'].mean():.2f}, std={arac['verimsizlik_skoru'].std():.2f}")


Arac master: 3,508

yas: min=1.0, max=19.0, mean=11.6
yakit_turu_cng: CNG=350 (10.0%)
verimsizlik_skoru: mean=22.27, std=27.85


---
## 3. egim_maruziyet (Q1) — Arac × Hat Egim Maruziyeti

Q1 sefer verisinden, her aracin gittigi hatlarin egim puanlari, sefer agirlikli ortalama.


In [3]:
# BOLUM 3: egim_maruziyet
# Sefer verisi (Q1 filter)
ah = pd.read_csv("../panel_data/temiz_veri/arac_gunluk_hatlar.csv", low_memory=False)
ah["TARIH"] = pd.to_datetime(ah["TARIH"], format="mixed")
ah_q1 = ah[ah["TARIH"] < SPLIT_DATE].copy()
print(f"arac_gunluk_hatlar Q1: {len(ah_q1):,} kayit")

# Hat egim verisi (Analiz 3 metodolojisi)
with open("../panel_data/hat_elevation.json", encoding="utf-8") as f:
    he_raw = _json.load(f)
he = pd.DataFrame([
    {"HATKODU": k, "rakim": v.get("rakım_farkı", 0), "tirm": v.get("tırmanma_m", 0)}
    for k, v in he_raw.items()
])
def mm_norm(s, q=None):
    if q is not None: s = s.clip(upper=s.quantile(q))
    mn, mx = s.min(), s.max()
    return ((s - mn)/(mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)
he["norm_r"] = mm_norm(he["rakim"], q=0.99)
he["norm_t"] = mm_norm(he["tirm"], q=0.99)
he["egim_puan"] = (he["norm_r"]*0.4 + he["norm_t"]*0.6).round(1)

# Sefer × egim merge
ah_q1 = ah_q1.merge(he[["HATKODU","egim_puan"]], on="HATKODU", how="left")
ah_q1_valid = ah_q1.dropna(subset=["egim_puan"])
print(f"Egim eslesen Q1 sefer: {len(ah_q1_valid):,} / {len(ah_q1):,} (%{len(ah_q1_valid)/len(ah_q1)*100:.1f})")

# Arac basina sefer agirlikli egim
def w_avg(g):
    return np.average(g["egim_puan"], weights=g["SEFER_SAYISI"].clip(lower=0.01))
arac_egim = ah_q1_valid.groupby("KAPINO").apply(w_avg).reset_index(name="egim_maruziyet")
print(f"Egim maruziyet hesaplanan arac: {len(arac_egim):,}")

# Master arac'a merge
arac = arac.merge(arac_egim, on="KAPINO", how="left")
median_egim = arac["egim_maruziyet"].median()
arac["egim_maruziyet"] = arac["egim_maruziyet"].fillna(median_egim)
print(f"egim_maruziyet: mean={arac['egim_maruziyet'].mean():.2f}, NaN sonrasi median fill: {median_egim:.2f}")


arac_gunluk_hatlar Q1: 658,866 kayit
Egim eslesen Q1 sefer: 657,507 / 658,866 (%99.8)
Egim maruziyet hesaplanan arac: 6,717
egim_maruziyet: mean=48.46, NaN sonrasi median fill: 49.89


---
## 4. hat_zorluk (Q1) — Hat Bilesik Zorluk Skoru

Egim (30%) + Uzunluk (20%) + Ariza yogunlugu (30%) + Trafik (20%)
Sonra arac × hat agirlikli ortalama.


In [4]:
# BOLUM 4: hat_zorluk (Analiz 9 metodolojisi)
# Hat profili
hat_ariza = df_q1.groupby("HATKODU").agg(
    hat_ort_skor=("ciddiyet_skoru","mean"),
    hat_uzunluk=("HATUZUNLUK","mean"),
).reset_index()
hat_sefer = ah_q1.groupby("HATKODU").agg(
    sefer_top=("SEFER_SAYISI","sum"),
    n_gun=("TARIH","nunique"),
).reset_index()
hat_sefer["gunluk_sefer"] = hat_sefer["sefer_top"] / hat_sefer["n_gun"].clip(lower=1)

hat = hat_ariza.merge(hat_sefer, on="HATKODU", how="left").merge(he[["HATKODU","egim_puan"]], on="HATKODU", how="left").fillna(0)
hat["n_egim"] = norm_0_100(hat["egim_puan"])
hat["n_uzun"] = norm_0_100(hat["hat_uzunluk"])
hat["n_ariza"] = norm_0_100(hat["hat_ort_skor"])
hat["n_trafik"] = norm_0_100(hat["gunluk_sefer"])
hat["hat_zorluk"] = (hat["n_egim"]*0.30 + hat["n_uzun"]*0.20 + hat["n_ariza"]*0.30 + hat["n_trafik"]*0.20).round(1)

print(f"Hat profili: {len(hat):,} hat, ort hat_zorluk={hat['hat_zorluk'].mean():.1f}")

# Arac × hat (sefer agirlikli)
ah_q1_zor = ah_q1.merge(hat[["HATKODU","hat_zorluk"]], on="HATKODU", how="left")
arac_hz = ah_q1_zor.dropna(subset=["hat_zorluk"]).groupby("KAPINO").apply(
    lambda g: np.average(g["hat_zorluk"], weights=g["SEFER_SAYISI"].clip(lower=0.01))
).reset_index(name="hat_zorluk")
arac = arac.merge(arac_hz, on="KAPINO", how="left")
median_hz = arac["hat_zorluk"].median()
arac["hat_zorluk"] = arac["hat_zorluk"].fillna(median_hz)
print(f"hat_zorluk: mean={arac['hat_zorluk'].mean():.2f}, NaN fill median: {median_hz:.2f}")


Hat profili: 679 hat, ort hat_zorluk=30.1
hat_zorluk: mean=35.71, NaN fill median: 33.52


---
## 5. garaj_sistem_lift + garaj_marka_lift (Q1)

Garaj × Sistem ve Garaj × Marka lift skorlari. Bu garaj kalitesi gostergeleri.


In [5]:
# BOLUM 5: garaj_sistem_lift + garaj_marka_lift
genel_ciddi = df_q1["ciddi_ariza"].mean()
print(f"Q1 genel ciddi orani: {genel_ciddi:.3f}")

# garaj_sistem_lift: arac garajinin sistem bazli lift'i (arac × sistem agirlikli)
gsis = df_q1.groupby(["GARAJ","ARIZAUSTKODTANIM"]).agg(
    n=("ciddi_ariza","count"),
    sis_oran=("ciddi_ariza","mean"),
).reset_index()
gsis["lift"] = gsis["sis_oran"] / genel_ciddi

# Arac sistem profili (Q1)
arac_sis = df_q1.groupby(["KAPINO","ARIZAUSTKODTANIM"]).size().reset_index(name="n_ariza")
arac_sis = arac_sis.merge(df_q1[["KAPINO","GARAJ"]].drop_duplicates("KAPINO"), on="KAPINO", how="left")
arac_sis = arac_sis.merge(gsis[["GARAJ","ARIZAUSTKODTANIM","lift"]], on=["GARAJ","ARIZAUSTKODTANIM"], how="left")
arac_sis["lift"] = arac_sis["lift"].fillna(1.0)
arac_sis["agirlikli"] = arac_sis["lift"] * arac_sis["n_ariza"]
arac_gsl = arac_sis.groupby("KAPINO").apply(
    lambda g: g["agirlikli"].sum() / g["n_ariza"].sum() if g["n_ariza"].sum() > 0 else 1.0
).reset_index(name="garaj_sistem_lift")
arac = arac.merge(arac_gsl, on="KAPINO", how="left")
arac["garaj_sistem_lift"] = arac["garaj_sistem_lift"].fillna(1.0)
print(f"garaj_sistem_lift: mean={arac['garaj_sistem_lift'].mean():.3f}, range=[{arac['garaj_sistem_lift'].min():.3f}, {arac['garaj_sistem_lift'].max():.3f}]")

# garaj_marka_lift: garaj × marka cifti icin lift
gml = df_q1.groupby(["GARAJ","MARKA"]).agg(
    m_oran=("ciddi_ariza","mean"),
).reset_index()
gml["garaj_marka_lift"] = gml["m_oran"] / genel_ciddi
arac = arac.merge(gml[["GARAJ","MARKA","garaj_marka_lift"]], on=["GARAJ","MARKA"], how="left")
arac["garaj_marka_lift"] = arac["garaj_marka_lift"].fillna(1.0)
print(f"garaj_marka_lift: mean={arac['garaj_marka_lift'].mean():.3f}, range=[{arac['garaj_marka_lift'].min():.3f}, {arac['garaj_marka_lift'].max():.3f}]")


Q1 genel ciddi orani: 0.370
garaj_sistem_lift: mean=0.967, range=[0.000, 2.702]
garaj_marka_lift: mean=0.961, range=[0.394, 2.702]


---
## 6. cascade_risk_skor (Q1) — Aynı Aracta 24h Tekrar Arıza Orani

Analiz 1 metodolojisi: Q1 icinde, arac × ardisik ariza, 24h icinde tekrar var mi?


In [6]:
# BOLUM 6: cascade_risk_skor
df_q1_sort = df_q1.sort_values(["KAPINO","OLAYTARIHI"]).reset_index(drop=True)
df_q1_sort["arac_prev_time"] = df_q1_sort.groupby("KAPINO")["OLAYTARIHI"].shift(1)
df_q1_sort["sure_gec_saat"] = (df_q1_sort["OLAYTARIHI"] - df_q1_sort["arac_prev_time"]).dt.total_seconds() / 3600
df_q1_sort["cascade_24h"] = (df_q1_sort["sure_gec_saat"] <= 24).astype(int)

cas_arac = df_q1_sort.groupby("KAPINO")["cascade_24h"].mean().reset_index(name="cascade_risk_skor")
arac = arac.merge(cas_arac, on="KAPINO", how="left")
arac["cascade_risk_skor"] = arac["cascade_risk_skor"].fillna(0)
print(f"cascade_risk_skor: mean={arac['cascade_risk_skor'].mean():.3f}, range=[{arac['cascade_risk_skor'].min():.3f}, {arac['cascade_risk_skor'].max():.3f}]")


cascade_risk_skor: mean=0.106, range=[0.000, 0.667]


---
## 7. farkli_sofor_sayisi (Q1) — Sofor Cesitliligi

Arac basina Q1 doneminde kac farkli sofor kullanmis?


In [7]:
# BOLUM 7: farkli_sofor_sayisi
sof_q1 = df_q1.dropna(subset=["SOFOR_SICILNO"]).copy()
sof_q1["SOFOR_SICILNO"] = sof_q1["SOFOR_SICILNO"].astype(str)
arac_sof = sof_q1.groupby("KAPINO")["SOFOR_SICILNO"].nunique().reset_index(name="farkli_sofor_sayisi")
arac = arac.merge(arac_sof, on="KAPINO", how="left")
arac["farkli_sofor_sayisi"] = arac["farkli_sofor_sayisi"].fillna(0).astype(int)
print(f"farkli_sofor_sayisi: mean={arac['farkli_sofor_sayisi'].mean():.1f}, range=[{arac['farkli_sofor_sayisi'].min()}, {arac['farkli_sofor_sayisi'].max()}]")


farkli_sofor_sayisi: mean=6.9, range=[0, 31]


---
## 8. dur_kalk_index (Q1) — Mekanik Stres Indeksi

Formul: KAPASITE × sefer_q1 / ort_sefer_km_q1
Yani: yuk × yogunluk × (kisa hat) = dur-kalk yipranma.


In [8]:
# BOLUM 8: dur_kalk_index
# Q1 sefer ve KM toplami (arac_gunluk_hatlar uzerinden basit, kesin km icin sefer_temiz gerek)
# Basit yaklasim: GUZERGAHUZUNLUK kullan (sefer_temiz cok buyuk)
# Alternatif: arac_gunluk_hatlar SEFER_SAYISI × HAT_UZUNLUK
ah_q1_uzun = ah_q1.merge(df_q1[["HATKODU","HATUZUNLUK"]].drop_duplicates("HATKODU"), on="HATKODU", how="left")
ah_q1_uzun["km_yapilan"] = ah_q1_uzun["SEFER_SAYISI"] * ah_q1_uzun["HATUZUNLUK"] / 1000  # metre→km

arac_km = ah_q1_uzun.groupby("KAPINO").agg(
    sefer_q1=("SEFER_SAYISI","sum"),
    km_q1=("km_yapilan","sum"),
).reset_index()
arac_km["ort_sefer_km_q1"] = (arac_km["km_q1"] / arac_km["sefer_q1"].clip(lower=1)).round(2)

# Master'a merge
arac = arac.merge(arac_km, on="KAPINO", how="left")
arac["sefer_q1"] = arac["sefer_q1"].fillna(0)
arac["ort_sefer_km_q1"] = arac["ort_sefer_km_q1"].fillna(arac["ort_sefer_km_q1"].median())
arac["KAPASITE"] = arac["KAPASITE"].replace(0, np.nan).fillna(arac["KAPASITE"].median())

arac["dur_kalk_index"] = (
    arac["KAPASITE"] * arac["sefer_q1"] / arac["ort_sefer_km_q1"].clip(lower=1)
).round(2)
print(f"dur_kalk_index: mean={arac['dur_kalk_index'].mean():.1f}, range=[{arac['dur_kalk_index'].min():.1f}, {arac['dur_kalk_index'].max():.1f}]")


dur_kalk_index: mean=3684.5, range=[0.0, 78628.9]


---
## 9. ariza_q1 + gecmis_ciddi_oran (Q1)

Direkt arıza sayim ve ciddi orani.


In [9]:
# BOLUM 9: ariza_q1 + gecmis_ciddi_oran
ariza_q1_stats = df_q1.groupby("KAPINO").agg(
    ariza_q1=("ciddi_ariza","count"),
    gecmis_ciddi_oran=("ciddi_ariza","mean"),
).reset_index()
arac = arac.merge(ariza_q1_stats, on="KAPINO", how="left")
arac["ariza_q1"] = arac["ariza_q1"].fillna(0).astype(int)
arac["gecmis_ciddi_oran"] = arac["gecmis_ciddi_oran"].fillna(0)
print(f"ariza_q1: mean={arac['ariza_q1'].mean():.1f}, range=[{arac['ariza_q1'].min()}, {arac['ariza_q1'].max()}]")
print(f"gecmis_ciddi_oran: mean={arac['gecmis_ciddi_oran'].mean():.3f}, range=[{arac['gecmis_ciddi_oran'].min():.3f}, {arac['gecmis_ciddi_oran'].max():.3f}]")
print()
print(f"Q1'de hic ariza yapmamis arac: {(arac['ariza_q1']==0).sum()} ({(arac['ariza_q1']==0).mean()*100:.1f}%)")


ariza_q1: mean=7.9, range=[0, 31]
gecmis_ciddi_oran: mean=0.350, range=[0.000, 1.000]

Q1'de hic ariza yapmamis arac: 63 (1.8%)


---
## 9b. INTERACTION FEATURES (V6.2 yeni)

V6.1'de tespit edilen problem: Anadolu Mercedes (19 yas, 5-13 ciddi arıza) gibi yasli yipranmis araclar **kacirildi (186 FN)**.

**Bilimsel cozum:** garaj_marka_lift dominant olmasına ragmen `yas × ciddi` etkilesimi modele acikca verilir.

- `yas_x_gecmis_ciddi` = yas × ciddi_oran (yaslı + yüksek ciddi oran)
- `yas_x_ariza_q1` = yas × Q1 toplam ariza
- `yasli_yipranmis_flag` = (yas >= 15 AND ciddi_oran >= 0.35) flag

In [10]:
# BOLUM 9b: INTERACTION FEATURES (V6.2 yas x ciddi etkilesimi)
# Bilimsel gerekce: V6.1'de Anadolu Mercedes (19 yas) yaslilar kacirildi
# garaj_marka_lift dominant + yas ayri feature -> interaction yok
# Yeni: explicit yas x ciddi etkilesimi

arac['yas_x_gecmis_ciddi'] = arac['yas'] * arac['gecmis_ciddi_oran']
arac['yas_x_ariza_q1'] = arac['yas'] * arac['ariza_q1']
arac['yasli_yipranmis_flag'] = ((arac['yas'] >= 15) & (arac['gecmis_ciddi_oran'] >= 0.35)).astype(int)

print('=== V6.2 INTERACTION FEATURES ===')
print(f'yas_x_gecmis_ciddi:    mean={arac["yas_x_gecmis_ciddi"].mean():.2f}, range=[{arac["yas_x_gecmis_ciddi"].min():.2f}, {arac["yas_x_gecmis_ciddi"].max():.2f}]')
print(f'yas_x_ariza_q1:        mean={arac["yas_x_ariza_q1"].mean():.1f}, range=[{arac["yas_x_ariza_q1"].min():.0f}, {arac["yas_x_ariza_q1"].max():.0f}]')
print(f'yasli_yipranmis_flag:  toplam={arac["yasli_yipranmis_flag"].sum()} arac (yas>=15 & ciddi_oran>=0.35)')

# Anadolu Mercedes 19 yas kontrol
anadolu_mer = arac[(arac['GARAJ']=='Anadolu') & (arac['MARKA']=='MERCEDES') & (arac['yas']>=15)]
print(f'\nAnadolu Mercedes 15+ yas arac: {len(anadolu_mer)}')
if len(anadolu_mer) > 0:
    print(f'  Ortalama yas_x_gecmis_ciddi: {anadolu_mer["yas_x_gecmis_ciddi"].mean():.2f}')
    print(f'  yasli_yipranmis_flag toplami: {anadolu_mer["yasli_yipranmis_flag"].sum()}/{len(anadolu_mer)}')

=== V6.2 INTERACTION FEATURES ===
yas_x_gecmis_ciddi:    mean=4.14, range=[0.00, 19.00]
yas_x_ariza_q1:        mean=92.8, range=[0, 468]
yasli_yipranmis_flag:  toplam=374 arac (yas>=15 & ciddi_oran>=0.35)

Anadolu Mercedes 15+ yas arac: 263
  Ortalama yas_x_gecmis_ciddi: 5.65
  yasli_yipranmis_flag toplami: 107/263


---
## 9c. ZAMAN-TREND FEATURES (V6.5.2 yeni)

**Sorun:** V6.5'te model arıza geçmişini sadece TOPLAM sayılar olarak görüyor (`ariza_q1`, `gecmis_ciddi_oran`). Mart'a doğru arıza hızlanıyor mu, son ayda ne oldu — bunu göremiyor.

**Çözüm:** 3 zaman-trend feature ekle (multicollinearity'e dikkat):

1. **`son_30_gun_ariza`** — Mart ayı arıza sayısı (recency sinyali)
2. **`son_ay_orani`** — `son_30_gun_ariza / ariza_q1` (oran şeklinde, multicollinearity'i kırar)
3. **`ariza_ivme`** — `(Mart+0.5) / (Ocak+0.5) - 1` (Laplace smoothing α=0.5)

**Bilimsel gerekçe:**
- Genç AKIA P=1.0 saturated (kalibrasyon problemi). Trend feature kalibrasyonu yayar.
- M5732 V6.5'te ORTA — trend hızlanan ise KRİTİK'e atlayabilir.

In [11]:
# BOLUM 9c: ZAMAN-TREND FEATURES (V6.5.2 yeni)
OCAK_END = pd.Timestamp("2025-02-01")
MART_START = pd.Timestamp("2025-03-01")

# 1. Mart ayi arızalar
mart_ariza = df_q1[df_q1["OLAYTARIHI"] >= MART_START].groupby("KAPINO").size()
arac["son_30_gun_ariza"] = arac["KAPINO"].map(mart_ariza).fillna(0).astype(int)

# 2. Son ay orani (multicollinearity'i kirar)
arac["son_ay_orani"] = arac["son_30_gun_ariza"] / arac["ariza_q1"].clip(lower=1)
arac["son_ay_orani"] = arac["son_ay_orani"].clip(upper=1.0)

# 3. Ariza ivme (Laplace smoothing α=0.5)
ocak_ariza = df_q1[df_q1["OLAYTARIHI"] < OCAK_END].groupby("KAPINO").size()
arac["_ocak_tmp"] = arac["KAPINO"].map(ocak_ariza).fillna(0).astype(int)
arac["ariza_ivme"] = ((arac["son_30_gun_ariza"] + 0.5) / (arac["_ocak_tmp"] + 0.5) - 1).round(3)
arac = arac.drop(columns=["_ocak_tmp"])

print('=== V6.5.2 ZAMAN-TREND FEATURES ===')
print(f'son_30_gun_ariza:  mean={arac["son_30_gun_ariza"].mean():.2f}, range=[{arac["son_30_gun_ariza"].min()}, {arac["son_30_gun_ariza"].max()}]')
print(f'son_ay_orani:      mean={arac["son_ay_orani"].mean():.3f}, range=[{arac["son_ay_orani"].min():.3f}, {arac["son_ay_orani"].max():.3f}]')
print(f'ariza_ivme:        mean={arac["ariza_ivme"].mean():.3f}, range=[{arac["ariza_ivme"].min():.2f}, {arac["ariza_ivme"].max():.2f}]')

# Multicollinearity hizli kontrol (target hesaplanmadan once)
print('\n=== MULTICOLLINEARITY (yeni x eski) ===')
risk_found = False
for new in ['son_30_gun_ariza', 'son_ay_orani', 'ariza_ivme']:
    for old in ['ariza_q1', 'gecmis_ciddi_oran', 'cascade_risk_skor']:
        r = arac[[new, old]].corr().iloc[0, 1]
        flag = ' RISK' if abs(r) > 0.85 else ''
        if abs(r) > 0.85: risk_found = True
        print(f'  {new:20s} x {old:25s} r={r:+.3f}{flag}')

if risk_found:
    print('\n!!! UYARI: Multicollinearity riski yuksek (r > 0.85)')
    print('    V6.5.2 modelinde feature elenmeli')
else:
    print('\nMulticollinearity makul, V6.5.2 ile devam edilebilir')

=== V6.5.2 ZAMAN-TREND FEATURES ===
son_30_gun_ariza:  mean=2.72, range=[0, 17]
son_ay_orani:      mean=0.340, range=[0.000, 1.000]
ariza_ivme:        mean=0.916, range=[-0.97, 26.00]

=== MULTICOLLINEARITY (yeni x eski) ===
  son_30_gun_ariza     x ariza_q1                  r=+0.745
  son_30_gun_ariza     x gecmis_ciddi_oran         r=+0.111
  son_30_gun_ariza     x cascade_risk_skor         r=+0.355
  son_ay_orani         x ariza_q1                  r=+0.036
  son_ay_orani         x gecmis_ciddi_oran         r=+0.035
  son_ay_orani         x cascade_risk_skor         r=+0.012
  ariza_ivme           x ariza_q1                  r=-0.023
  ariza_ivme           x gecmis_ciddi_oran         r=+0.030
  ariza_ivme           x cascade_risk_skor         r=+0.080

Multicollinearity makul, V6.5.2 ile devam edilebilir


---
## 10. target_q2 (Q2) — Hedef Değişken (≥3 ciddi arıza eşiği)

**V6 Final tanım:** Araç başına Q2 döneminde **3 veya daha fazla** ciddi arıza (geniş ciddi tanımı: 4 SONUCTIPI kategorisi).

**Neden ≥3 eşik?** (TARGET_KARSILASTIRMA.ipynb sonucu — 4 alternatif test edildi)
- Pozitif oran: **%54** (dengeli — class_weight gereksiz)
- Ortalama |r|=0.221 (en yüksek, 4 alternatif arasında)
- yakit_turu_cng restore: r=-0.118 (CNG koruyucu — A7 ile tutarlı)
- ARACCINSI Cramer V 0.285 (KORUKLU/SOLO ayrımı korunuyor)
- Q3 transfer sağlam: eşik sabit, mevsim çarpanı yoğunluğu post-prediction düzeltir
- Operasyonel anlam: 90 günde 3+ ciddi = ayda 1+ = sorunlu araç

In [12]:
# BOLUM 10: target_q2 (≥3 ciddi eşiği)
target = df_q2.groupby("KAPINO").agg(
    q2_ariza_n=("ciddi_ariza","count"),
    q2_ciddi_n=("ciddi_ariza","sum"),
).reset_index()
arac = arac.merge(target, on="KAPINO", how="left")
arac["q2_ariza_n"] = arac["q2_ariza_n"].fillna(0).astype(int)
arac["q2_ciddi_n"] = arac["q2_ciddi_n"].fillna(0).astype(int)

# TARGET: Q2'de ≥3 ciddi arıza
arac["target_q2"] = (arac["q2_ciddi_n"] >= 3).astype(int)

print(f"target_q2 (≥3 ciddi eşiği) dağılım:")
print(arac["target_q2"].value_counts().sort_index())
print(f"\nPozitif oran: {arac['target_q2'].mean()*100:.1f}%")
print(f"Q2'de hiç arıza yapmamış araç: {(arac['q2_ariza_n']==0).sum()} ({(arac['q2_ariza_n']==0).mean()*100:.1f}%)")
print(f"Q2'de arıza yapmış ama <3 ciddi (target=0): {((arac['q2_ariza_n']>0) & (arac['target_q2']==0)).sum()}")
print(f"\nq2_ciddi_n dağılımı:")
print(arac["q2_ciddi_n"].describe().round(1))

target_q2 (≥3 ciddi eşiği) dağılım:
target_q2
0    1615
1    1893
Name: count, dtype: int64

Pozitif oran: 54.0%
Q2'de hiç arıza yapmamış araç: 56 (1.6%)
Q2'de arıza yapmış ama <3 ciddi (target=0): 1559

q2_ciddi_n dağılımı:
count    3508.0
mean        3.4
std         2.9
min         0.0
25%         1.0
50%         3.0
75%         5.0
max        18.0
Name: q2_ciddi_n, dtype: float64


---
## 11. Feature Matrix — Final Bilgisayar

12 feature + meta + target birlestir, NaN'lari kontrol et.


In [13]:
# BOLUM 11: Feature matrix final
FEATURES = [
    "yas", "egim_maruziyet", "garaj_sistem_lift", "garaj_marka_lift",
    "yakit_turu_cng", "verimsizlik_skoru", "hat_zorluk", "cascade_risk_skor",
    "farkli_sofor_sayisi", "dur_kalk_index", "ariza_q1", "gecmis_ciddi_oran",
]

META = ["KAPINO", "GARAJ", "MARKA", "MODEL", "ARACCINSI", "YAKITTURU"]
TARGET = ["target_q2"]
EXTRA = ["q2_ariza_n", "q2_ciddi_n", "MODELYILI", "KAPASITE", "tuketim_100km", "sefer_q1", "km_q1", "ort_sefer_km_q1",
         "yas_x_gecmis_ciddi", "yas_x_ariza_q1", "yasli_yipranmis_flag",  # V6.2 interaction
         "son_30_gun_ariza", "son_ay_orani", "ariza_ivme"]  # V6.5.2 trend

final = arac[META + FEATURES + TARGET + EXTRA].copy()
print(f"Final feature matrix: {len(final):,} arac × {len(final.columns)} kolon")
print()
print("=== NaN KONTROL ===")
for c in final.columns:
    n = final[c].isna().sum()
    if n > 0:
        print(f"  {c:25s} NaN: {n}")
    else:
        print(f"  {c:25s} OK")
print()
print("=== FEATURE ISTATISTIK ===")
print(final[FEATURES].describe().round(3).T)


Final feature matrix: 3,508 arac × 33 kolon

=== NaN KONTROL ===
  KAPINO                    OK
  GARAJ                     OK
  MARKA                     OK
  MODEL                     OK
  ARACCINSI                 OK
  YAKITTURU                 OK
  yas                       OK
  egim_maruziyet            OK
  garaj_sistem_lift         OK
  garaj_marka_lift          OK
  yakit_turu_cng            OK
  verimsizlik_skoru         OK
  hat_zorluk                OK
  cascade_risk_skor         OK
  farkli_sofor_sayisi       OK
  dur_kalk_index            OK
  ariza_q1                  OK
  gecmis_ciddi_oran         OK
  target_q2                 OK
  q2_ariza_n                OK
  q2_ciddi_n                OK
  MODELYILI                 OK
  KAPASITE                  OK
  tuketim_100km             OK
  sefer_q1                  OK
  km_q1                     NaN: 10
  ort_sefer_km_q1           OK
  yas_x_gecmis_ciddi        OK
  yas_x_ariza_q1            OK
  yasli_yipranmis_flag      OK


---
## 12. Sanity Check — Korelasyon ve Hata Kontrolu

Feature'larin target ile korelasyonu, multicollinearity, mantik kontrolu.


In [14]:
# BOLUM 12: Sanity check
from scipy import stats
print("=== FEATURE × TARGET KORELASYON ===")
print(f"{'Feature':25s} {'Pearson r':>12s} {'p-value':>12s}")
print("-"*55)
for f in FEATURES:
    r, p = stats.pearsonr(final[f].fillna(0), final["target_q2"])
    p_str = f"{p:.4f}" if p > 0.0001 else "<0.0001"
    print(f"{f:25s} {r:>+12.4f} {p_str:>12s}")

print()
print("=== TARGET POZITIF/NEGATIF DAGILIMI ===")
pos = final[final["target_q2"]==1]
neg = final[final["target_q2"]==0]
print(f"Pozitif (target=1): {len(pos):,} ({len(pos)/len(final)*100:.1f}%)")
print(f"Negatif (target=0): {len(neg):,} ({len(neg)/len(final)*100:.1f}%)")

print()
print("=== POZITIF vs NEGATIF FEATURE ORTALAMASI ===")
for f in FEATURES:
    m1 = pos[f].mean()
    m0 = neg[f].mean()
    fark = m1 - m0
    print(f"  {f:25s} pos={m1:>8.2f}  neg={m0:>8.2f}  fark={fark:>+7.2f}")

print()
print("=== Q1 BOS ARAC KONTROL ===")
bos_q1 = final[final["ariza_q1"]==0]
print(f"Q1'de hic ariza yapmamis: {len(bos_q1)} arac")
print(f"  -> Bunlarin target_q2 dagilimi:")
print(bos_q1["target_q2"].value_counts())
print(f"  Pozitif orani: {bos_q1['target_q2'].mean()*100:.1f}% (genel {final['target_q2'].mean()*100:.1f}%)")


=== FEATURE × TARGET KORELASYON ===
Feature                      Pearson r      p-value
-------------------------------------------------------
yas                            +0.1218      <0.0001
egim_maruziyet                 +0.1582      <0.0001
garaj_sistem_lift              +0.1678      <0.0001
garaj_marka_lift               +0.3232      <0.0001
yakit_turu_cng                 -0.1181      <0.0001
verimsizlik_skoru              +0.3255      <0.0001
hat_zorluk                     +0.3471      <0.0001
cascade_risk_skor              +0.1580      <0.0001
farkli_sofor_sayisi            +0.3716      <0.0001
dur_kalk_index                 -0.0577       0.0006
ariza_q1                       +0.3534      <0.0001
gecmis_ciddi_oran              +0.1503      <0.0001

=== TARGET POZITIF/NEGATIF DAGILIMI ===
Pozitif (target=1): 1,893 (54.0%)
Negatif (target=0): 1,615 (46.0%)

=== POZITIF vs NEGATIF FEATURE ORTALAMASI ===
  yas                       pos=   12.08  neg=   10.96  fark=  +1.13
  egim_

---
## 13. CSV Export — features_final_v6_q1.csv

V6 modeline doğrudan girecek hazır CSV.


In [15]:
# BOLUM 13: CSV export
out_path = "features_final_v6_q1.csv"
final.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Export: {out_path}")
print(f"  {len(final):,} arac × {len(final.columns)} kolon")
print(f"\nKolonlar:")
for c in final.columns:
    print(f"  - {c}")

print()
print("=== ARAC SAYI DOGRULAMA ===")
print(f"Final arac: {len(final):,}")
print(f"ariza_model unique: {df['KAPINO'].nunique():,}")
print(f"Q1 arac: {df_q1['KAPINO'].nunique():,}")
print(f"Q2 arac: {df_q2['KAPINO'].nunique():,}")

print()
print("=== V6 ML NOTEBOOK ICIN HAZIR ===")
print(f"Feature listesi: {FEATURES}")
print(f"Target: target_q2 (binary)")
print(f"Pozitif oran: {final['target_q2'].mean()*100:.1f}%")
print(f"\nSonraki adim: ML_MODEL_V6.ipynb")


Export: features_final_v6_q1.csv
  3,508 arac × 33 kolon

Kolonlar:
  - KAPINO
  - GARAJ
  - MARKA
  - MODEL
  - ARACCINSI
  - YAKITTURU
  - yas
  - egim_maruziyet
  - garaj_sistem_lift
  - garaj_marka_lift
  - yakit_turu_cng
  - verimsizlik_skoru
  - hat_zorluk
  - cascade_risk_skor
  - farkli_sofor_sayisi
  - dur_kalk_index
  - ariza_q1
  - gecmis_ciddi_oran
  - target_q2
  - q2_ariza_n
  - q2_ciddi_n
  - MODELYILI
  - KAPASITE
  - tuketim_100km
  - sefer_q1
  - km_q1
  - ort_sefer_km_q1
  - yas_x_gecmis_ciddi
  - yas_x_ariza_q1
  - yasli_yipranmis_flag
  - son_30_gun_ariza
  - son_ay_orani
  - ariza_ivme

=== ARAC SAYI DOGRULAMA ===
Final arac: 3,508
ariza_model unique: 3,508
Q1 arac: 3,445
Q2 arac: 3,452

=== V6 ML NOTEBOOK ICIN HAZIR ===
Feature listesi: ['yas', 'egim_maruziyet', 'garaj_sistem_lift', 'garaj_marka_lift', 'yakit_turu_cng', 'verimsizlik_skoru', 'hat_zorluk', 'cascade_risk_skor', 'farkli_sofor_sayisi', 'dur_kalk_index', 'ariza_q1', 'gecmis_ciddi_oran']
Target: target_